In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# 動的回路の実行

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';



<Accordion>
<AccordionItem title="パッケージ・バージョン">

このページのコードは、以下の要件を使用して開発されました。
これらのバージョン以降の使用をお勧めします。

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>

動的回路は、量子回路の実行中に Qubit を中間測定し、その測定結果に基づいて回路内で古典的な論理演算を実行できる強力なツールです。このプロセスは _古典フィードフォワード_ とも呼ばれます。動的回路を最大限に活用する方法の理解はまだ初期段階ですが、量子研究コミュニティはすでに以下のような多くのユースケースを特定しています。

* 効率的な量子状態準備（[GHZ 状態](https://journals.aps.org/prxquantum/abstract/10.1103/PRXQuantum.5.030339)、[W 状態](https://arxiv.org/abs/2403.07604)（W 状態の詳細については["State preparation by shallow circuits using feed forward"](https://arxiv.org/abs/2307.14840)も参照）、および広いクラスの[行列積状態](https://arxiv.org/abs/2404.16083)）
* 浅い回路を使用した同じチップ上の Qubit 間の[効率的な長距離エンタングルメント](https://journals.aps.org/prxquantum/abstract/10.1103/PRXQuantum.5.030339)
* [IQP に類似した回路の効率的なサンプリング](https://arxiv.org/pdf/2505.04705)

ただし、動的回路によるこれらの改善はトレードオフを伴います。中間回路測定と古典演算は通常 2-Qubit ゲートよりも長い実行時間を必要とし、この時間の増加が回路深度の削減による利点を相殺する可能性があります。そのため、IBM Quantum&reg; が動的回路の[新バージョン](/announcements/product-updates/2025-03-03-new-version-dynamic-circuits)をリリースするにあたり、中間回路測定の長さの削減が改善の焦点となっています。動的回路を使用する際のその他の制限については、[Estimator](/guides/estimator-options#options-compatibility-table) または [Sampler](/guides/sampler-options#options-compatibility-table) の機能互換性テーブルを参照してください。

[OpenQASM 3 仕様](https://openqasm.com/language/classical.html#looping-and-branching)ではいくつかの制御フロー構造が定義されていますが、Qiskit Runtime は現在、条件付き `if` 文のみをサポートしています。Qiskit SDK では、これは [QuantumCircuit](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit) の [if_test](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#if_test) メソッドに対応しています。このメソッドは[コンテキスト・マネージャー](https://docs.python.org/3/reference/datamodel.html#with-statement-context-managers)を返し、通常は `with` 文で使用されます。このガイドでは、この条件文の使用方法を説明します。

> **Note:** このガイドのコード例では、中間回路測定に標準の measure 命令を使用しています。しかし、バックエンドがサポートしている場合は、代わりに `MidCircuitMeasure` 命令を使用することをお勧めします。詳細については、[中間回路測定セクション](#midcircuit)を参照してください。

## 動的回路をサポートするバックエンドの検索
アカウントがアクセスでき、動的回路をサポートするすべてのバックエンドを見つけるには、以下のようなコードを実行してください。この例では、[ログイン認証情報を保存済み](/guides/save-credentials)であることを前提としています。または、Qiskit Runtime サービス・アカウントを初期化する際に[明示的に認証情報を指定](/guides/initialize-account#explicit)することもできます。これにより、特定のインスタンスやプラン・タイプで利用可能なバックエンドを表示できます。

> **Note:** - アカウントで利用可能なバックエンドは、認証情報で指定されたインスタンスによって異なります。
> - 動的回路の新バージョンは現在、すべてのバックエンドのすべてのユーザーに利用可能です。詳細については、[アナウンス](/announcements/product-updates/2025-09-25-new-dynamic-circuits)を参照してください。

In [ ]:
# This cell is hidden from users. It hides all those "...instance was not set..." warnings.
import warnings

warnings.filterwarnings("ignore", message=".*Instance was not set*")

In [2]:
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
dc_backends = service.backends(dynamic_circuits=True)
print(dc_backends)

[<IBMBackend('ibm_pittsburgh')>, <IBMBackend('ibm_kingston')>, <IBMBackend('ibm_marrakesh')>, <IBMBackend('ibm_fez')>, <IBMBackend('ibm_boston')>]


<span id="midcircuit"></span>
## Mid-circuit measurements

Prior to `qiskit-ibm-runtime` v0.43.0, `measure` was the only measurement instruction in Qiskit. Mid-circuit measurements, however, have different tuning requirements than _terminal_ measurements (measurements that happen at the end of a circuit). For example, you need to consider the instruction duration when tuning a mid-circuit measurement because longer instructions cause noisier circuits. You don't need to consider instruction duration for terminal measurements because there are no instructions after terminal measurements.

<Admonition type="note">
The `MidCircuitMeasure` instruction maps to the `measure_2` instruction reported in the backend's `supported_instructions`. However,  `measure_2` is not supported on all backends. Use `service.backends(filters=lambda b: "measure_2" in b.supported_instructions)` to find backends that support it.  New measurements might be added in the future, but this is not guarenteed.
</Admonition>

### `MidCircuitMeasure` method

In `qiskit-ibm-runtime` v0.43.0, the `MidCircuitMeasure` instruction was introduced. As the name suggests, it is a new measurement instruction that is optimized for mid-circuit on IBM&reg; QPUs.  While you can use `QuantumCircuit.measure` for a mid-circuit measurement, because of its design, `MidCircuitMeasure` is typically a better choice.  For example, it adds less overhead to your circuit than when using `QuantumCircuit.measure`.

In [3]:
from qiskit import QuantumCircuit
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime.circuit import MidCircuitMeasure

service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, dynamic_circuits=True
)

circ = QuantumCircuit(2, 2)
circ.x(0)
circ.append(MidCircuitMeasure(), [0], [0])
# circ.measure([0], [0])
# circ.measure_all()
print(circ.draw(cregbundle=False))

     ┌───┐┌────────────┐
q_0: ┤ X ├┤0           ├
     └───┘│            │
q_1: ─────┤  Measure_2 ├
          │            │
c_0: ═════╡0           ╞
          └────────────┘
c_1: ═══════════════════
                        


<span id="midcircuit"></span>

## 中間回路測定

`qiskit-ibm-runtime` v0.43.0 より前は、`measure` が Qiskit における唯一の測定命令でした。しかし、中間回路測定は_最終_測定（回路の終わりで行われる測定）とは異なるチューニング要件を持っています。例えば、中間回路測定をチューニングする際には命令の継続時間を考慮する必要があります。継続時間が長いほど回路のノイズが増加するからです。最終測定の後には命令がないため、最終測定では命令の継続時間を考慮する必要はありません。

> **Note:** `MidCircuitMeasure` 命令は、バックエンドの `supported_instructions` で報告される `measure_2` 命令にマッピングされます。ただし、`measure_2` はすべてのバックエンドでサポートされているわけではありません。`service.backends(filters=lambda b: "measure_2" in b.supported_instructions)` を使用して、サポートするバックエンドを見つけてください。将来、新しい測定が追加される可能性がありますが、これは保証されていません。

### `MidCircuitMeasure` メソッド

`qiskit-ibm-runtime` v0.43.0 では、`MidCircuitMeasure` 命令が導入されました。名前が示すように、これは IBM&reg; QPU 上の中間回路測定に最適化された新しい測定命令です。中間回路測定に `QuantumCircuit.measure` を使用することもできますが、そのの設計上、`MidCircuitMeasure` は通常より良い選択です。例えば、`QuantumCircuit.measure` を使用する場合よりも回路へのオーバーヘッドが少なくなります。

In [4]:
from qiskit_ibm_runtime import SamplerV2, QiskitRuntimeService
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, dynamic_circuits=True
)

# Create a dynamic circuit

qubits = QuantumRegister(1)
clbits = ClassicalRegister(1)
qc = QuantumCircuit(qubits, clbits)
(q0,) = qubits
(c0,) = clbits

qc.h(q0)
qc.measure(q0, c0)
with qc.if_test((c0, 1)):
    qc.x(q0)
qc.measure(q0, c0)


# Convert to an ISA circuit for the given backend

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

# Generate samplers for backend targets
sampler = SamplerV2(backend)

# Submit jobs
sampler_job = sampler.run([isa_circuit])
result = sampler_job.result()

print(
    f">>> {' Job ID:':<10}  {sampler_job.job_id()} ({sampler_job.status()})"
)

>>>  Job ID:    d88cakp789is7391vq0g (DONE)


## Qiskit Runtime limitations

Be aware of the following constraints when running dynamic circuits in Qiskit Runtime.

- Due to the limited physical memory on control electronics, there is also a limit on the number of `if` statements and size of their operands. This limit is a function of the number of broadcasts and the number of broadcasted bits in a job (not a circuit).

   When processing an `if` condition, measurement data needs to be transferred to the control logic to make that evaluation. A broadcast is a transfer of unique classical data, and broadcasted bits is the number of classical bits being transferred. Consider the following:

   ```python
   c0 = ClassicalRegister(3)
   c1 = ClassicalRegister(5)
   ...
   with circuit.if_test((c0, 1)) ...
   with circuit.if_test((c0, 3)) ...
   with circuit.if_test((c1[2], 1)) ...
   ```
   In the previous code example, the first two `if_test` objects on `c0` are considered one broadcast because the content of `c0` did not change, and thus does not need to be re-broadcasted. The `if_test` on `c1` is a second broadcast. The first one broadcasts all three bits in `c0` and the second broadcasts just one bit, making a total of four broadcasted bits.

   Currently, if you broadcast 60 bits each time, then the job can have approximately 300 broadcasts. If you broadcast just one bit each time, however, then the job can have 2400 broadcasts.

- The operand used in an `if_test` statement must be 32 or fewer bits. Thus, if you are comparing an entire `ClassicalRegister`, the size of that `ClassicalRegister` must be 32 or fewer bits. If you are comparing just a single bit from a `ClassicalRegister`, however, that `ClassicalRegister` can be of any size (since the operand is only one bit).

   For example, the "Not valid" code block does not work because `cr` is more than 32 bits.  You can, however, use a classical register wider than 32 bits if you are testing only one bit, as shown in the "Valid" code block.

   <Tabs>
   <TabItem value="Not valid" label="Not valid">
      ```python
         cr = ClassicalRegister(50)
         qr = QuantumRegister(50)
         circuit = QuantumCircuit(qr, cr)
         ...
         circ.measure(qr, cr)
         with circ.if_test((cr, 15)):
            ...
      ```
   </TabItem>
   <TabItem value="Valid" label="Valid">
      ```python
         cr = ClassicalRegister(50)
         qr = QuantumRegister(50)
         circuit = QuantumCircuit(qr, cr)
         ...
         circ.measure(qr, cr)
         with circ.if_test((cr[5], 1)):
            ...
      ```
   </TabItem>
   </Tabs>

- Nested conditionals are not allowed. For example, the following code block will not work because it has an `if_test` inside another `if_test`:
   <Tabs>
    <TabItem value="Not valid" label="Not valid">
        ```python
           c1 = ClassicalRegister(1, "c1")
           c2 = ClassicalRegister(2, "c2")
           ...
           with circ.if_test((c1, 1)):
            with circ.if_test(c2, 1)):
             ...
        ```
     </TabItem>
     <TabItem value="Valid" label="Valid">
        ```python
        cr = ClassicalRegister(2)
        ...
        with circuit.if_test((cr, 0b11)):
          ...
        ```
     </TabItem>
    </Tabs>

- Having `reset` or measurements inside conditionals is not supported.
- Arithmetic operations are not supported.
- See the [OpenQASM 3 feature table](/docs/guides/qasm-feature-table) to determine which OpenQASM 3 features are supported on Qiskit and Qiskit Runtime.
- When OpenQASM 3 (instead of `QuantumCircuit`) is used as the input format to pass circuits to Qiskit Runtime primitives, only instructions that can be loaded into Qiskit are supported. Classical operations, for example, are not supported because they cannot be loaded into Qiskit. See [Import an OpenQASM 3 program into Qiskit](/docs/guides/interoperate-qiskit-qasm3#import-an-openqasm-3-program-into-qiskit) for more information.
- The `for`, `while`, and `switch` instructions are not supported.

## Use dynamic circuits with Estimator

Since Estimator does not support dynamic circuits, you can use [Sampler](/docs/guides/get-started-with-sampler) and build your own measurement circuits instead.

To replicate Estimator's behavior, follow this process:

1. Group the terms of all observables into a partition.  This can be done by using the [`PauliList` API](/docs/api/qiskit/qiskit.quantum_info.PauliList#group_qubit_wise_commuting), for example.
     <Admonition type="note">
      You can use the [`BitArray`](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.primitives.BitArray#expectation_values) primitive attribute to compute the expectation values of the provided observables.
     </Admonition>
2. Execute one basis change circuit per partition (whichever basis change needs to be done for each partition). See the Measurement bases addon utility [`measurement_bases` module](https://qiskit.github.io/qiskit-addon-utils/apidocs/qiskit_addon_utils.exp_vals.html#qiskit_addon_utils.exp_vals.get_measurement_bases) for more information. For more information, see the [documentation](https://qiskit.github.io/qiskit-addon-utils/) for the Qiskit addon utilities package.
3. Add back together the results for each partition.

## Restrictions

Review any [Feature compatibility table](/docs/guides/estimator-options#options-compatibility-table) to understand restrictions when using dynamic circuits. Note that feature compatibility is not primitive-dependent.

## Next steps

<Admonition type="tip" title="Recommendations">
- Learn how to implement accurate dynamic decoupling by using [stretch](/docs/guides/stretch).
- Review the [classical feedforward and control flow](/docs/guides/classical-feedforward-and-control-flow) guide.
- Use [circuit schedule visualization](/docs/guides/qiskit-runtime-circuit-timing) to debug and optimize your dynamic circuits.
- Not all functions are compatible with dynamic circuits. Check the feature compatibility section for [Sampler](/docs/guides/sampler-options#feature-compatibility) or [Executor](/docs/guides/executor-options#feature-compatibility) for details.
</Admonition>